In [1]:
import os
import time
import shutil
import datetime
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from typing import List, Optional, Callable
from concurrent.futures import ProcessPoolExecutor, as_completed

# import baostock as bs
# from baostock.data.resultset import ResultData
# from baostock_utils import baostock_login_context, baostock_relogin, baostock_login
import yfinance as yf
from yahooquery import Ticker

from qlib_dump_bin import DumpDataAll
import qlib

In [ ]:
nasdaq_stocks = pd.read_csv("https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt", sep="|")

nasdaq_symbols = nasdaq_stocks["Symbol"].tolist()

print(f"{len(nasdaq_symbols)} stocks in total")

In [13]:
nasdaq_symbols = [s for s in nasdaq_symbols if isinstance(s, str) and s.strip()]
stock_data = yf.download(nasdaq_symbols, start="2020-01-01", end="2024-03-01")

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  4815 of 4817 completed
[*********************100%***********************]  4816 of 4817 completed
3561 Failed downloads:
['BIAFW', 'EUDAW', 'ASPCR', 'GOVXW', 'LVROW', 'HYMCW', 'STSSW', 'XAGEW', 'BTMWW', 'BNAIW', 'SDSTW', 'OSRHW', 'MSSAW', 'HSPOW', 'CHARR', 'KPLTW', 'KITTW', 'RCT', 'CLSKW', 'OUSTW', 'DFLIW', 'XBPEW', 'CINGW', 'BETRW', 'NXLIW', 'ADBG', 'CRMLW', 'CORZZ', 'CXAIW', 'ECDAW', 'MSPRZ', 'KVACW', 'SONDW', 'CIFRW', 'TDACW', 'SXTPW', 'PLMKW', 'ANSCW', 'VSTEW', 'SLXNW', 'VACHW', 'GIGGW', 'SAIHW', 'OUSTZ', 'CYCUW', 'ARKOW', 'ACONW', 'INVZW', 'FFAIW', 'MDCXW', 'DSYWW', 'XOSWW', 'LEXXW', 'LTRYW', 'AUROW', 'GBBKR', 'PANG', 'SVREW', 'OABIW', 'CRGOW', 'SWVLW', 'ATMVR', 'IVCAW', 'OACCW', 'MSSAR', 'DRMAW', 'DECAW', 'LOTWW', 'BEAGR', 'ASBPW', 'OCSAW', 'CDROW', 'NRXPW', 'POLEW', 'BNZIW', 'BLDEW', 'CAPNR', 'MVSTW', 'BTBDW', 'TBLAW', 'ONMDW', 'ISRLW', 'SHOTW', 'SYTAW', 'TNONW', 'TAVIR', 'EMCGR', 'LNZAW', 'BSLKW', 'YHNAR', 'GDSTR', 'OCEAW', 'S

In [14]:
stock_data = stock_data.dropna(axis=1, how="all")

In [ ]:
dfs = {}
valid_symbols = []
for t in nasdaq_symbols:
    try:
        df = stock_data.xs(t, axis=1, level='Ticker')
        dfs[t] = df
        valid_symbols.append(t)
    except Exception as e:
        pass

export_path = "../../export/"
os.makedirs(export_path, exist_ok=True)

for ticker, df in dfs.items():
    file_path = os.path.join(export_path, f"{ticker}.csv")
    df = df.reset_index()
    df.to_csv(file_path, index=False)


In [ ]:
csv_folder = "../../export/"  
qlib_output_folder = "../../qlib_format_data"

os.makedirs(qlib_output_folder, exist_ok=True)

for file in os.listdir(csv_folder):
    if file.endswith(".csv"):
        file_path = os.path.join(csv_folder, file)
        symbol = file.replace(".csv", "")

        df = pd.read_csv(file_path)
        df.rename(columns={"Date": "date", "Open": "open", "High": "high", 
                           "Low": "low", "Close": "close", "Volume": "volume"}, inplace=True)
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")

        df = df[["date", "open", "high", "low", "close", "volume"]]

        output_path = os.path.join(qlib_output_folder, f"{symbol}.csv")
        df.to_csv(output_path, index=False)

In [ ]:
import shutil
from pathlib import Path
from qlib_dump_bin import DumpDataAll

def csv_to_qlib(
    csv_dir: str,
    qlib_dir: str,
    exclude_fields: str = "date,code",
    symbol_field_name: str = "code"
):


    csv_dir = Path(csv_dir).expanduser().resolve()
    qlib_dir = Path(qlib_dir).expanduser().resolve()

    export_dir = csv_dir / "export"
    export_dir.mkdir(exist_ok=True)


    for file in csv_dir.glob("*.csv"):
        code = file.stem 

        df = pd.read_csv(file)

        df["code"] = code.upper()

        out_file = export_dir / f"{code.upper()}.csv"
        df.to_csv(out_file, index=False)

    DumpDataAll(
        csv_path=str(export_dir),
        qlib_dir=str(qlib_dir),
        max_workers=1,
        exclude_fields="date,symbol,code",
        symbol_field_name="symbol"
    ).dump()

    cal_dir = qlib_dir / "calendars"
    day_txt = cal_dir / "day.txt"
    day_future_txt = cal_dir / "day_future.txt"
    if day_txt.exists():
        shutil.copy(day_txt, day_future_txt)

if __name__ == "__main__":
    csv_folder = "../qlib_format_data/export"
    qlib_data_dir = "../.qlib/qlib_data/stock_data_2024h1"

    csv_to_qlib(csv_dir=csv_folder, qlib_dir=qlib_data_dir)


In [4]:
def initialize_qlib(qlib_data_path) -> None:
    import qlib
    from qlib.config import REG_CN
    qlib.init(provider_uri=qlib_data_path, region=REG_CN)
    global _QLIB_INITIALIZED
    _QLIB_INITIALIZED = True

In [5]:
initialize_qlib('D:/Desktop/.qlib/qlib_data/cn_data_2024h1')

[46272:MainThread](2025-03-21 13:18:00,562) INFO - qlib.Initialization - [config.py:420] - default_conf: client.
[46272:MainThread](2025-03-21 13:18:01,330) INFO - qlib.Initialization - [__init__.py:74] - qlib successfully initialized based on client settings.
[46272:MainThread](2025-03-21 13:18:01,331) INFO - qlib.Initialization - [__init__.py:76] - data_path={'__DEFAULT_FREQ': WindowsPath('D:/Desktop/.qlib/qlib_data/cn_data_2024h1')}


In [ ]:
from qlib.data import D
from qlib.data.dataset.loader import QlibDataLoader

ticker = 'SH600519'
start_date = "2023-01-01"
end_date = "2024-01-01"

df = D.features([ticker], fields=["$close", "$open", "$high", "$low", "$volume"], start_time=start_date, end_time=end_date)

print(df.head())

                            $close        $open        $high         $low  \
instrument datetime                                                         
SH600519   2023-01-03  2875.875244  2877.853516  2889.872314  2835.979004   
           2023-01-04  2867.563477  2875.858643  2890.321045  2852.585693   
           2023-01-05  2993.885254  2887.495117  2993.885254  2880.845703   
           2023-01-06  2998.489990  3002.396484  3012.004883  2970.612305   
           2023-01-09  3060.711426  3050.405029  3075.306885  3005.222412   

                           $volume  
instrument datetime                 
SH600519   2023-01-03  1566087.875  
           2023-01-04  1228128.750  
           2023-01-05  2884047.500  
           2023-01-06  1498108.625  
           2023-01-09  1863464.625  


In [ ]:
QlibDataLoader(config=['$close', '$open', '$high']).load('all', "2023-01-01", "2023-01-05")

In [ ]:
csi300_stocks = D.list_instruments(
    instruments={"market": "all",
                 "filter_pipe": []
    },
    start_time="2020-01-01",
    end_time="2021-01-01"
)
print(csi300_stocks)


{'AAL': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'AAON': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'AAPL': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'AAXJ': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'ABCS': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'ABOS': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'ABP': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'ACAD': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'ACGL': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'ACGLO': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'ACHV': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'ACIU': [(Timestamp('2020-01-02 00:00:00'), Timestamp('2021-01-01 00:00:00'))], 'ACIW': [(Timestamp('2020-01-02 00:00:00